# Hybrid Quantum Brain Tumor Detection Training

This notebook separates each logical step into its own cell: imports, configuration, quantum circuit, model builder, dataset loader, training utilities, and the run logic.

In [9]:
import os
# Suppress TensorFlow, absl and other noisy logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # INFO and WARNING from C++ backend
os.environ['ABSL_MIN_LOG_LEVEL'] = '2'

import warnings
# Ignore Python warnings (broad) to reduce notebook noise
warnings.filterwarnings('ignore')

import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

# Import numerical and ML libs
import numpy as np
import tensorflow as tf
import pennylane as qml

tf.get_logger().setLevel('ERROR')
try:
    tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
except Exception:
    pass

# Filter specific TensorFlow warnings that still leak through during epochs
class _TfWarningFilter(logging.Filter):
    def filter(self, record):
        blocked = [
            'casting an input of type complex128',
            'tf.placeholder is deprecated',
            'TensorFlow GPU support is not available on native Windows',
        ]
        msg = record.getMessage()
        return not any(text in msg for text in blocked)

tf.get_logger().addFilter(_TfWarningFilter())

# Try to silence absl if present
try:
    import absl.logging
    absl.logging.set_verbosity(absl.logging.ERROR)
except Exception:
    pass

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight


In [10]:
# Configuration
N_QUBITS = 8
N_QUANTUM_LAYERS = 3
INPUT_FEATURES = 16
BATCH_SIZE = 64
EPOCHS = 100
SAMPLES = 4096
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)


In [12]:
# Quantum device and circuit definition using qubits and gates
dev = qml.device("default.qubit", wires=N_QUBITS)

def quantum_circuit(inputs, weights):
    """Quantum circuit for the QNN layer.

    - Encodes classical features into qubits
    - Applies parameterized single-qubit rotations
    - Adds entanglement with CNOT gates
    - Measures Pauli-Z expectation values
    """
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation="Y")

    for layer in range(N_QUANTUM_LAYERS):
        for qubit in range(N_QUBITS):
            qml.RX(weights[layer, qubit, 0], wires=qubit)
            qml.RY(weights[layer, qubit, 1], wires=qubit)
            qml.RZ(weights[layer, qubit, 2], wires=qubit)

        for qubit in range(N_QUBITS - 1):
            qml.CNOT(wires=[qubit, qubit + 1])
        qml.CNOT(wires=[N_QUBITS - 1, 0])

    return [qml.expval(qml.PauliZ(wires=i)) for i in range(N_QUBITS)]

weight_shapes = {"weights": (N_QUANTUM_LAYERS, N_QUBITS, 3)}
qnode = qml.QNode(quantum_circuit, dev, interface="tf", diff_method="backprop")

# Lightweight TensorFlow wrapper layer for the QNode (fallback when PennyLane's
# KerasLayer is not available in the installed pennylane version).
class QuantumLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.weights_var = self.add_weight(
            name="weights",
            shape=weight_shapes["weights"],
            initializer=tf.random_normal_initializer(seed=RANDOM_SEED),
            trainable=True,
            dtype=tf.float32,
        )
        super().build(input_shape)

    def call(self, inputs):
        res = qnode(inputs, self.weights_var)
        res = tf.cast(tf.math.real(tf.stack(res, axis=1)), tf.float32)
        return res

qlayer = QuantumLayer()


In [13]:
# Hybrid model builder: classical layers + quantum layer + classical output
def build_hybrid_qnn_model():
    classical_input = tf.keras.Input(shape=(INPUT_FEATURES,), name="classical_features")

    x = tf.keras.layers.Dense(64, activation="relu", name="dense_pre_quantum")(classical_input)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(N_QUBITS, activation="linear", name="dense_to_quantum")(x)

    x = qlayer(x)

    x = tf.keras.layers.Dense(32, activation="relu", name="dense_post_quantum")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    output = tf.keras.layers.Dense(1, activation="sigmoid", name="prediction")(x)

    model = tf.keras.Model(inputs=classical_input, outputs=output, name="hybrid_qnn")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


In [14]:
# Synthetic dataset helper (replace with actual MRI-derived features for real training)
def load_synthetic_brain_tumor_data(samples=SAMPLES):
    rng = np.random.default_rng(RANDOM_SEED)
    X = rng.normal(size=(samples, INPUT_FEATURES))
    y = rng.integers(0, 2, size=(samples, 1)).astype(np.float32)

    # Make the synthetic task learnable: add a stronger class-dependent offset to half the features
    half = INPUT_FEATURES // 2
    # Increase separability for the positive class and reduce noise
    X[y.flatten() == 1, :half] += 2.0
    X += rng.normal(scale=0.05, size=X.shape)  # small gaussian noise

    return X, y

# Optional real dataset loader stub
def load_dataset_from_folder(folder_path=r"C:\Users\PREMRAJ\Desktop\Dataset", feature_count=INPUT_FEATURES):
    """Load brain MRI feature vectors from a folder or precomputed file.

    This function should be replaced with actual preprocessing of MRI scans.
    """
    raise NotImplementedError(
        "Replace load_dataset_from_folder() with real MRI feature extraction logic."
    )


In [15]:
# Training loop and save utilities
def train_hybrid_qnn_model(model, X_train, y_train, X_val, y_val, class_weight=None):
    callbacks = []
    callbacks.append(tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True))
    callbacks.append(tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6))
    callbacks.append(tf.keras.callbacks.ModelCheckpoint("hybrid_quantum_model_best.h5", monitor="val_loss", save_best_only=True))

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        callbacks=callbacks,
        class_weight=class_weight,
        verbose=2,
    )
    return history

def save_model(model, path="hybrid_quantum_model.h5"):
    model.save(path)
    print(f"Saved hybrid quantum model to {path}")

In [16]:
# Main execution cell
def main():
    print("Preparing hybrid quantum model training...")
    X, y = load_synthetic_brain_tumor_data(samples=SAMPLES)

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
    )

    # Compute class weights to handle any imbalance
    y_flat = y_train.flatten().astype(int)
    classes = np.unique(y_flat)
    class_weights = compute_class_weight('balanced', classes=classes, y=y_flat)
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, class_weights)}

    model = build_hybrid_qnn_model()
    model.summary()

    history = train_hybrid_qnn_model(model, X_train, y_train, X_val, y_val, class_weight=class_weight_dict)

    save_model(model)

    final_acc = history.history["val_accuracy"][-1]
    print(f"Final validation accuracy: {final_acc:.4f}")


In [17]:
if __name__ == "__main__":
    main()

Preparing hybrid quantum model training...


Model: "hybrid_qnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ classical_features (InputLayer) │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_pre_quantum (Dense)       │ (None, 64)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_to_quantum (Dense)        │ (None, 8)              │           520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ quantum_layer_1 (QuantumLayer)  │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_post_quantum (Dense)      │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ prediction (Dense)              │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,385 (9.32 KB)

 Trainable params: 2,193 (8.57 KB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/100
52/52 - 25s - 477ms/step - accuracy: 0.4933 - loss: 0.8595 - val_accuracy: 0.4439 - val_loss: 0.7474 - learning_rate: 5.0000e-04
Epoch 2/100
52/52 - 4s - 82ms/step - accuracy: 0.4881 - loss: 0.8324 - val_accuracy: 0.4622 - val_loss: 0.7184 - learning_rate: 5.0000e-04
Epoch 3/100
52/52 - 4s - 71ms/step - accuracy: 0.5058 - loss: 0.7962 - val_accuracy: 0.4744 - val_loss: 0.7021 - learning_rate: 5.0000e-04
Epoch 4/100
52/52 - 4s - 73ms/step - accuracy: 0.4985 - loss: 0.7961 - val_accuracy: 0.4902 - val_loss: 0.6979 - learning_rate: 5.0000e-04
Epoch 5/100
52/52 - 4s - 69ms/step - accuracy: 0.5082 - loss: 0.7783 - val_accuracy: 0.4988 - val_loss: 0.6972 - learning_rate: 5.0000e-04
Epoch 6/100
52/52 - 4s - 76ms/step - accuracy: 0.5027 - loss: 0.7779 - val_accuracy: 0.5159 - val_loss: 0.6897 - learning_rate: 5.0000e-04
Epoch 7/100
52/52 - 4s - 72ms/step - accuracy: 0.5055 - loss: 0.7710 - val_accuracy: 0.5537 - val_loss: 0.6821 - learning_rate: 5.0000e-04
Epoch 8/100
52/52 - 4s - 